**COVID-19 Data Analysis & Dashboard System**

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
import plotly.express as px

**1. Data Collection**

*Load COVID-19 dataset from Kaggle*

In [2]:
df = pd.read_csv("covid_19_clean_complete.csv")

In [3]:
print("Shape:", df.shape)
print("\nColumn Names:")
print(df.columns)
df

Shape: (49068, 10)

Column Names:
Index(['Province/State', 'Country/Region', 'Lat', 'Long', 'Date', 'Confirmed',
       'Deaths', 'Recovered', 'Active', 'WHO Region'],
      dtype='str')


,Province/State,Country/Region,Lat,Long,Date,Confirmed,Deaths,Recovered,Active,WHO Region
0,NaN,Afghanistan,33.939110,67.709953,2020-01-22,0,0,0,0,Eastern Mediterranean
1,NaN,Albania,41.153300,20.168300,2020-01-22,0,0,0,0,Europe
2,NaN,Algeria,28.033900,1.659600,2020-01-22,0,0,0,0,Africa
3,NaN,Andorra,42.506300,1.521800,2020-01-22,0,0,0,0,Europe
4,NaN,Angola,-11.202700,17.873900,2020-01-22,0,0,0,0,Africa
...,...,...,...,...,...,...,...,...,...,...
49063,NaN,Sao Tome and Principe,0.186400,6.613100,2020-07-27,865,14,734,117,Africa
49064,NaN,Yemen,15.552727,48.516388,2020-07-27,1691,483,833,375,Eastern Mediterranean
49065,NaN,Comoros,-11.645500,43.333300,2020-07-27,354,7,328,19,Africa
49066,NaN,Tajikistan,38.861000,71.276100,2020-07-27,7235,60,6028,1147,Europe


*Select only required columns for analysis*

In [4]:
cols_to_drop = [
    'Province/State',
    'Lat',
    'Long',
    'Last Update',
    'Incident Rate',
    'Case Fatality Ratio'
]

df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

print("Updated Columns:", df.columns)
df

Updated Columns: Index(['Country/Region', 'Date', 'Confirmed', 'Deaths', 'Recovered', 'Active',
       'WHO Region'],
      dtype='str')


,Country/Region,Date,Confirmed,Deaths,Recovered,Active,WHO Region
0,Afghanistan,2020-01-22,0,0,0,0,Eastern Mediterranean
1,Albania,2020-01-22,0,0,0,0,Europe
2,Algeria,2020-01-22,0,0,0,0,Africa
3,Andorra,2020-01-22,0,0,0,0,Europe
4,Angola,2020-01-22,0,0,0,0,Africa
...,...,...,...,...,...,...,...
49063,Sao Tome and Principe,2020-07-27,865,14,734,117,Africa
49064,Yemen,2020-07-27,1691,483,833,375,Eastern Mediterranean
49065,Comoros,2020-07-27,354,7,328,19,Africa
49066,Tajikistan,2020-07-27,7235,60,6028,1147,Europe


**2. Data Cleaning & Preprocessing**

*Convert Date column into proper format*

In [5]:
df['Date'] = pd.to_datetime(df['Date'])
print(df['Date'])

0       2020-01-22
1       2020-01-22
2       2020-01-22
3       2020-01-22
4       2020-01-22
           ...    
49063   2020-07-27
49064   2020-07-27
49065   2020-07-27
49066   2020-07-27
49067   2020-07-27
Name: Date, Length: 49068, dtype: datetime64[us]


*Remove duplicate records*

In [6]:
print(df.isnull().sum())
df.fillna(df.mean(numeric_only=True), inplace=True)

df = df.drop_duplicates()

Country/Region    0
Date              0
Confirmed         0
Deaths            0
Recovered         0
Active            0
WHO Region        0
dtype: int64


**3. Descriptive Statistics**

*Total Confirmed Cases*

In [7]:
print("Total Confirmed Caes\n")
print(df['Confirmed'].sum())

Total Confirmed Caes

828507763


*Total Deaths*

In [8]:
print("Total Deaths\n")
print(df['Deaths'].sum())

Total Deaths

43384902


*Total Recoveries*

In [9]:
print("Total Recoveries\n")
print(df['Recovered'].sum())

Total Recoveries

388408160


*Total Active Cases*

In [10]:
print("Total Active Cases\n")
print(df['Active'].sum())

Total Active Cases

396714701


**4. COVID Trend Analysis**

In [11]:
global_trend = df.groupby('Date')[['Confirmed', 'Deaths', 'Recovered', 'Active']].sum().reset_index()

fig = px.line(global_trend, x='Date', y=['Confirmed', 'Deaths', 'Recovered', 'Active'],
              title='Global COVID-19 Trend Over Time',
              labels={'value':'Number of Cases', 'variable':'Case Type'})
fig.show()

*Analyze confirmed cases over time*

In [12]:
global_confirmed = df.groupby('Date')['Confirmed'].sum().reset_index()

fig = px.line(global_confirmed, x='Date', y='Confirmed',
              title='Global Confirmed COVID-19 Cases Over Time',
              labels={'Confirmed':'Total Confirmed Cases', 'Date':'Date'},
              template='plotly_white')
fig.show()

*Identify peak periods (waves)*

In [13]:
global_confirmed = df.groupby('Date')['Confirmed'].sum().reset_index()

global_confirmed['Daily_New'] = global_confirmed['Confirmed'].diff().fillna(0)

mean_daily = global_confirmed['Daily_New'].mean()
std_daily = global_confirmed['Daily_New'].std()

global_confirmed['Peak'] = global_confirmed['Daily_New'] > (mean_daily + 1.5*std_daily)

fig = px.line(global_confirmed, x='Date', y='Daily_New', title='Global Daily New Cases with Peaks Highlighted',
              labels={'Daily_New':'Daily New Cases', 'Date':'Date'},
              template='plotly_white')
fig.show()

*Study growth trends*

In [14]:
global_confirmed = df.groupby('Date')['Confirmed'].sum().reset_index()

global_confirmed['Daily_New'] = global_confirmed['Confirmed'].diff().fillna(0)

global_confirmed['Daily_Growth_Rate'] = global_confirmed['Confirmed'].pct_change().fillna(0) * 100

fig = px.line(global_confirmed, x='Date', y='Daily_Growth_Rate',
              title='Global Daily Growth Rate of Confirmed Cases (%)',
              labels={'Daily_Growth_Rate':'Growth Rate (%)', 'Date':'Date'},
              template='plotly_white')
fig.show()

**5. Country-Based Analysis**

*Identify most affected countries*

In [15]:
latest_date = df['Date'].max()
latest_data = df[df['Date'] == latest_date]

top_countries = latest_data.sort_values(by='Confirmed', ascending=False).head(10)

fig = px.bar(top_countries, x='WHO Region', y='Confirmed',
             title='Top 10 Most Affected Countries',
             text='Confirmed', color='Confirmed', color_continuous_scale='Reds')
fig.show()

*Comparing Countries*

In [16]:
latest_date = df['Date'].max()

latest_data = df[df['Date'] == latest_date]

country_comparison = latest_data[['WHO Region','Confirmed','Deaths','Recovered']]

country_comparison = country_comparison.sort_values(by='Confirmed', ascending=False)

country_comparison

,WHO Region,Confirmed,Deaths,Recovered
49030,Americas,4290259,148011,1325804
48835,Americas,2442375,87618,1846641
48936,South-East Asia,1480073,33408,951166
48992,Europe,816680,13334,602249
49005,Africa,452529,7067,274925
...,...,...,...,...
48918,Europe,7,0,6
49048,Americas,5,0,0
49060,Europe,4,0,1
49052,Europe,3,0,3


*Top 10 Countries by Confirmed Cases*

In [17]:
fig = px.bar(country_comparison.head(10), x='WHO Region', y='Confirmed',
             text='Confirmed', color='Confirmed', color_continuous_scale='Reds',
             title='Top 10 Countries by Confirmed COVID-19 Cases')
fig.show()

*Top 10 Countries by Deaths*

In [18]:
fig = px.bar(country_comparison.sort_values('Deaths', ascending=False).head(10),
             x='WHO Region', y='Deaths', text='Deaths', color='Deaths',
             color_continuous_scale='gray', title='Top 10 Countries by Deaths')
fig.show()

*Top 10 Countries by Recoveries*

In [19]:
fig = px.bar(country_comparison.sort_values('Recovered', ascending=False).head(10),
             x='WHO Region', y='Recovered', text='Recovered', color='Recovered',
             color_continuous_scale='greens', title='Top 10 Countries by Recoveries')
fig.show()

**6. Dashboard - Based Analysis**

In [20]:
from dash import Dash, dcc, html, Input, Output

# Initialize app
app = Dash(__name__)

app.layout = html.Div([

    html.H1("COVID-19 Dashboard", style={'textAlign':'center'}),

    html.Br(),

    # Country Dropdown
    html.Div([
        html.Label("Select Country"),
        dcc.Dropdown(
            id='country-dropdown',
            options=[{'label': c, 'value': c} for c in sorted(df['Country/Region'].unique())],
            value='India',
            clearable=False
        )
    ], style={'width':'40%', 'margin':'auto'}),

    html.Br(),

    # Date Picker
    html.Div([
        html.Label("Select Date Range"),
        dcc.DatePickerRange(
            id='date-picker',
            min_date_allowed=df['Date'].min(),
            max_date_allowed=df['Date'].max(),
            start_date=df['Date'].min(),
            end_date=df['Date'].max()
        )
    ], style={'width':'40%', 'margin':'auto'}),

    html.Br(),

    # Charts
    dcc.Graph(id='line-chart'),
    dcc.Graph(id='bar-chart'),
    dcc.Graph(id='pie-chart'),
    dcc.Graph(id='scatter-chart'),
    dcc.Graph(id='heatmap-chart')

])

In [21]:
@app.callback(
    [Output('line-chart', 'figure'),
     Output('bar-chart', 'figure'),
     Output('pie-chart', 'figure'),
     Output('scatter-chart', 'figure'),
     Output('heatmap-chart', 'figure')],

    [Input('country-dropdown', 'value'),
     Input('date-picker', 'start_date'),
     Input('date-picker', 'end_date')]
)

def update_dashboard(selected_country, start_date, end_date):

    filtered_df = df[
        (df['Country/Region'] == selected_country) &
        (df['Date'] >= start_date) &
        (df['Date'] <= end_date)
    ].copy()

    latest_df = df[df['Date'] == df['Date'].max()]

    # 1️⃣ Line Chart (Cases over time)
    line_fig = px.line(
        filtered_df,
        x='Date',
        y=['Confirmed','Deaths','Recovered','Active'],
        title=f"{selected_country} COVID-19 Cases Over Time",
        template='plotly_dark'
    )

    # 2️⃣ Bar Chart (Top affected countries)
    top10 = latest_df.sort_values('Confirmed', ascending=False).head(10)

    bar_fig = px.bar(
        top10,
        x='WHO Region',
        y='Confirmed',
        color='Confirmed',
        title='Top 10 Affected Countries',
        template='plotly_dark'
    )

    # 3️⃣ Pie Chart (Case distribution)
    country_latest = latest_df[latest_df['Country/Region'] == selected_country].iloc[0]

    pie_fig = px.pie(
        values=[
            country_latest['Confirmed'],
            country_latest['Deaths'],
            country_latest['Recovered'],
            country_latest['Active']
        ],
        names=['Confirmed','Deaths','Recovered','Active'],
        title=f"{selected_country} Case Distribution"
    )

    # 4️⃣ Scatter Plot (Relationships)
    scatter_fig = px.scatter(
        latest_df,
        x='Confirmed',
        y='Deaths',
        size='Active',
        color='WHO Region',
        title='Confirmed vs Deaths',
        template='plotly_dark'
    )

    # 5️⃣ Heatmap (Correlation)
    numeric_cols = ['Confirmed','Deaths','Recovered','Active']
    corr_matrix = latest_df[numeric_cols].corr()

    heatmap_fig = px.imshow(
        corr_matrix,
        text_auto=True,
        color_continuous_scale='Viridis',
        title='COVID-19 Correlation Heatmap'
    )

    return line_fig, bar_fig, pie_fig, scatter_fig, heatmap_fig

In [23]:
if __name__ == "__main__":
    app.run(debug=True)

**7. Relationship Analysis**

*Confirmed vs Deaths*

In [ ]:
fig = px.scatter(df, x='Confirmed', y='Deaths', color='WHO Region',
                 hover_name='WHO Region',
                 title='Confirmed vs Deaths',
                 labels={'Confirmed':'Total Confirmed Cases', 'Deaths':'Total Deaths'},
                 template='plotly_white',
                 size='Deaths', size_max=15)
fig.show()

*Confirmed vs Recovered*

In [ ]:
fig = px.scatter(df, x='Confirmed', y='Recovered', color='WHO Region',
                 hover_name='WHO Region',
                 title='Confirmed vs Recovered',
                 labels={'Confirmed':'Total Confirmed Cases', 'Recovered':'Total Recovered Cases'},
                 template='plotly_white',
                 size='Recovered', size_max=15)
fig.show()

*Active vs Confirmed*

In [ ]:
df.loc[:, 'Active'] = df['Active'].apply(lambda x: max(0, x))
fig = px.scatter(df, x='Confirmed', y='Active', color='WHO Region',
                 hover_name='WHO Region',
                 title='Active Cases vs Confirmed Cases',
                 labels={'Confirmed':'Total Confirmed Cases', 'Active':'Active Cases'},
                 template='plotly_white',
                 size='Active', size_max=30)
fig.show()

*Analyze correlations*

In [ ]:
numeric_cols = ['Confirmed', 'Deaths', 'Recovered', 'Active']
corr_matrix = df[numeric_cols].corr()

fig = px.imshow(corr_matrix,
                text_auto=True,
                color_continuous_scale='Viridis',
                title='Correlation Matrix of COVID-19 Metrics')

fig.show()

**8. Data Visualization (Dashboard Charts)**

*Line chart → Cases over time*

In [ ]:
country = 'Americas'
df_country = df[df['WHO Region'] == country]

fig_line = px.line(df_country, x='Date', y=['Confirmed','Deaths','Recovered','Active'],
                   labels={'value':'Cases','variable':'Legend'},
                   title=f'{country}: COVID-19 Cases Over Time',
                   template='plotly_white')
fig_line.show()

*Bar chart → Top affected countries*

In [ ]:
latest_df = df[df['Date'] == df['Date'].max()]
top_countries = latest_df.sort_values('Confirmed', ascending=False).head(10)

fig_bar = px.bar(top_countries, x='WHO Region', y='Confirmed',
                 text='Confirmed', color='Confirmed',
                 color_continuous_scale='Reds',
                 title='Top 10 Affected Countries',
                 template='plotly_white')
fig_bar.show()

*Pie chart → Case distribution*

In [ ]:
latest_country = latest_df[latest_df['WHO Region'] == country].iloc[0]
values = [latest_country['Confirmed'], latest_country['Deaths'], latest_country['Recovered'], latest_country['Active']]
labels = ['Confirmed','Deaths','Recovered','Active']

fig_pie = px.pie(values=values, names=labels,
                 title=f'{country}: COVID-19 Case Distribution',
                 template='plotly_white')
fig_pie.show()

*Scatter plot → Relationships*

In [ ]:
fig_scatter = px.scatter(latest_df, x='Confirmed', y='Deaths',
                         size='Active', color='WHO Region',
                         hover_name='WHO Region',
                         title='Confirmed vs Deaths by Country',
                         template='plotly_white')
fig_scatter.show()

*Heatmap → Correlation*

In [ ]:
numeric_cols = ['Confirmed','Deaths','Recovered','Active']
corr_matrix = latest_df[numeric_cols].corr()

fig_heatmap = px.imshow(corr_matrix,
                        text_auto=True,
                        color_continuous_scale='Viridis',
                        title='Correlation of COVID-19 Metrics')
fig_heatmap.show()